# Long-Context Retrieval: Hard-Negative Patching

Tests whether the dual-encoder retrieval model degrades when each document is padded
with semantically similar (hard negative) passages.  

For every document `d_i` we:
1. Use the model's own embeddings + FAISS to find its **K nearest neighbours** (hard negatives).
2. Concatenate `d_i` with those neighbours → one longer "patched" document.
3. Re-embed the patched corpus and re-run retrieval.

We sweep K ∈ {0, 1, 3, 5} and report how metrics degrade.

In [ ]:
import os
import sys

# Adjust to local checkout path
project_dir = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
src_dir = os.path.join(project_dir, 'src')
os.chdir(project_dir)
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print('Project dir:', project_dir)

## 1. Load HeQ evaluation data

In [ ]:
from data.heq import HeQDatasetBuilder

builder = HeQDatasetBuilder()
eval_tasks = builder.build_eval_dataset(split='test', should_sample=True, filter_empty_answers=True)
task_data = eval_tasks['TASK_QUESTION_DOC']
print(f'Loaded {len(task_data)} (query, passage) pairs')
task_data[0]

In [ ]:
# Build a deduplicated passage corpus and record gold indices per query
passages = []         # unique passage texts
passage_to_idx = {}   # passage text → corpus index
queries = []          # query texts
gold_indices = []     # gold corpus index for each query

for item in task_data:
    q = item['anchor_text']
    p = item['positive_text']
    if p not in passage_to_idx:
        passage_to_idx[p] = len(passages)
        passages.append(p)
    queries.append(q)
    gold_indices.append(passage_to_idx[p])

print(f'Queries : {len(queries)}')
print(f'Passages: {len(passages)} (unique)')

## 2. Load model

In [ ]:
import torch
from transformers import AutoTokenizer
from model.dual_encoder.models import InfoNCEDualEncoder, InfoNCEDualEncoderConfig

# ── Set this to your checkpoint ──────────────────────────────────────────────
MODEL_PATH = 'outputs/models/dual_encoder/dual_encoder_infonce_heq/onlplab_alephbert-base/model/'
# ─────────────────────────────────────────────────────────────────────────────

config = InfoNCEDualEncoderConfig.from_pretrained(MODEL_PATH)
model  = InfoNCEDualEncoder.from_pretrained(MODEL_PATH, config=config)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device).eval()

tokenizer_q = AutoTokenizer.from_pretrained(config.query_model_name)
tokenizer_d = AutoTokenizer.from_pretrained(config.doc_model_name)

print('Device:', device)
print('Config:', config)

## 3. Encode queries and original passages

In [ ]:
from model.eval.eval_retrieval import batched_encode
import torch.nn.functional as F

BATCH_SIZE = 64
MAX_LENGTH = 512

q_emb = batched_encode(model=model, encoder=model.query_encoder,
                       tokenizer=tokenizer_q, texts=queries,
                       device=device, batch_size=BATCH_SIZE, max_length=MAX_LENGTH)
q_emb = F.normalize(q_emb.float(), p=2, dim=-1)

d_emb_orig = batched_encode(model=model, encoder=model.doc_encoder,
                             tokenizer=tokenizer_d, texts=passages,
                             device=device, batch_size=BATCH_SIZE, max_length=MAX_LENGTH)
d_emb_orig = F.normalize(d_emb_orig.float(), p=2, dim=-1)

print('q_emb:', q_emb.shape)
print('d_emb:', d_emb_orig.shape)

## 4. Retrieval metrics helper

In [ ]:
import numpy as np

def retrieval_metrics(q_emb: torch.Tensor, d_emb: torch.Tensor, gold_indices: list,
                      ks=(1, 5, 10)) -> dict:
    """Compute Accuracy@1, MRR, Recall@k for a list of gold indices."""
    sim = torch.matmul(q_emb, d_emb.t()).cpu().numpy()  # (Q, D)
    ranks = []
    for i, gi in enumerate(gold_indices):
        order = np.argsort(sim[i])[::-1]
        rank = int(np.where(order == gi)[0][0]) + 1  # 1-based
        ranks.append(rank)
    ranks = np.array(ranks)
    metrics = {
        'Acc@1'  : float(np.mean(ranks == 1)),
        'MRR'    : float(np.mean(1.0 / ranks)),
    }
    for k in ks:
        metrics[f'Recall@{k}'] = float(np.mean(ranks <= k))
    return metrics

## 5. Sweep K — patch corpus and evaluate

In [ ]:
from data.long_context.patch_documents import build_patched_corpus

K_VALUES = [0, 1, 3, 5]
results = {}

for k in K_VALUES:
    print(f'\n── K={k} ──')
    patched_texts, pos_positions = build_patched_corpus(
        documents=passages,
        embeddings=d_emb_orig,
        k=k,
        positive_position='random',
        seed=42,
    )
    avg_chars = np.mean([len(t) for t in patched_texts])
    print(f'  Avg patched doc length: {avg_chars:.0f} chars  '
          f'(was {np.mean([len(p) for p in passages]):.0f})')

    # Re-encode the patched corpus
    d_emb_patched = batched_encode(
        model=model, encoder=model.doc_encoder,
        tokenizer=tokenizer_d, texts=patched_texts,
        device=device, batch_size=BATCH_SIZE, max_length=MAX_LENGTH,
    )
    d_emb_patched = F.normalize(d_emb_patched.float(), p=2, dim=-1)

    metrics = retrieval_metrics(q_emb, d_emb_patched, gold_indices)
    results[k] = metrics
    for key, val in metrics.items():
        print(f'  {key:12s}: {val:.4f}')

## 6. Summary table

In [ ]:
import pandas as pd

df = pd.DataFrame(results).T
df.index.name = 'K (hard negatives)'
df.columns.name = 'Metric'
print(df.to_string(float_format=lambda x: f'{x:.4f}'))
df

## 7. Qualitative example (K=3)

Show a single query, its relevant passage, and the hard-negative passages it was bundled with.

In [ ]:
from data.long_context.patch_documents import find_hard_negatives, SEPARATOR

K_DEMO = 3
QUERY_IDX = 0   # change to inspect a different query

hard_neg_idx = find_hard_negatives(d_emb_orig, k=K_DEMO)

gi = gold_indices[QUERY_IDX]   # which passage is the gold
hn_indices = hard_neg_idx[gi]  # hard negatives for that passage

print('=== Query ===')
print(queries[QUERY_IDX])

print('\n=== Gold passage (positive) ===')
print(passages[gi])

for rank, hn in enumerate(hn_indices, 1):
    print(f'\n=== Hard negative {rank} (corpus idx {hn}) ===')
    print(passages[hn])

print('\n=== Patched document (positive first, then hard negatives) ===')
parts = [passages[gi]] + [passages[j] for j in hn_indices]
print(SEPARATOR.join(parts))

## 8. Effect of positive position

Does it matter where in the patch the genuine passage sits? (K=3)

In [ ]:
K_POS = 3
position_results = {}

for pos in ('first', 'last', 'random'):
    print(f'\n── positive_position={pos!r} ──')
    patched_texts, _ = build_patched_corpus(
        documents=passages, embeddings=d_emb_orig,
        k=K_POS, positive_position=pos, seed=42,
    )
    d_emb_p = batched_encode(
        model=model, encoder=model.doc_encoder,
        tokenizer=tokenizer_d, texts=patched_texts,
        device=device, batch_size=BATCH_SIZE, max_length=MAX_LENGTH,
    )
    d_emb_p = F.normalize(d_emb_p.float(), p=2, dim=-1)
    m = retrieval_metrics(q_emb, d_emb_p, gold_indices)
    position_results[pos] = m
    for key, val in m.items():
        print(f'  {key:12s}: {val:.4f}')

pd.DataFrame(position_results).T